In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [ ]:
def habr_search(queries, pages=1):
    if isinstance(queries, str):
        queries = [queries]

    rows = []

    for query in queries:
        query = query.replace(" ", "+")
        for page in range(1, pages + 1):
            url = f"https://habr.com/ru/search/page{page}/?q={query}"
            response = requests.get(url)
            soup = BeautifulSoup(response.text, "html.parser")

            articles = soup.find_all("article", class_="tm-articles-list__item")

            for article in articles:
                date = article.find("time").get("datetime")
                title = (
                    article.find("a", class_="tm-title__link").find("span").text.strip()
                )
                link = "https://habr.com" + article.find(
                    "a", class_="tm-title__link"
                ).get("href")
                rating = article.find(
                    "span", attrs={"data-test-id": "votes-meter-value"}
                )
                rating = rating.text if rating else 0

                nested_response = requests.get(link)
                nested_soup = BeautifulSoup(nested_response.text, "html.parser")
                description_body = nested_soup.find("div", class_="article-body")
                description = (
                    " ".join(description_body.stripped_strings)
                    if description_body
                    else None
                )

                rows.append(
                    {
                        "date": date,
                        "title": title,
                        "link": link,
                        "description": description,
                        "rating": rating,
                    }
                )

    res = pd.DataFrame(rows)
    return res.drop_duplicates(subset=["link"], keep="first")
